# EDA: English vs Hindi Language Use in VFT Responses

**Dataset:** `vft_responses.csv` — Verbal Fluency Task (VFT) responses from 35 subjects across 3 domains each.  
**Goal:** Explore patterns of English and Hindi/Hinglish language use, identify fully English-responding subjects, and analyse when subjects switch from Hindi/Hinglish to English.

---
**Columns:**
- `subject_id` — unique participant identifier
- `session_id` — session identifier
- `domain` — VFT category (`colours`, `animals`, `foods`, `body-parts`)
- `word` — word produced by the participant
- `rt_ms` — reaction time in milliseconds
- `position` — ordinal position of the word in the response sequence
- `language_type` — `English` or `Hindi/Hinglish`

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("merged_vft_spam_responses.csv")

# Normalise language column for convenience
df["is_hindi"] = df["language_type"] == "Hindi/Hinglish"

print("Dataset shape:", df.shape)
print("\nDomains      :", df["domain"].unique().tolist())
print("Language types:", df["language_type"].unique().tolist())
print("Total subjects:", df["subject_id"].nunique())
print("\nLanguage counts overall:")
print(df["language_type"].value_counts().to_string())
df.head()

Dataset shape: (1040, 10)

Domains      : ['colours', 'animals', 'foods', 'body-parts']
Language types: ['English', 'Hindi/Hinglish']
Total subjects: 35

Language counts overall:
language_type
Hindi/Hinglish    723
English           317


,subject_id,session_id,domain,position,word,language_type,rt_ms,x,y,is_hindi
0,10255,1qmxoH7jT7VECeLUVEKU,colours,1,red,English,2558.3,0.076446,1.003870,False
1,10255,1qmxoH7jT7VECeLUVEKU,colours,2,blue,English,1464.6,0.080923,0.540248,False
2,10255,1qmxoH7jT7VECeLUVEKU,colours,3,green,English,1505.6,0.074953,0.097112,False
3,10255,1qmxoH7jT7VECeLUVEKU,colours,4,indigo,English,1894.6,0.125355,0.534443,False
4,10255,1qmxoH7jT7VECeLUVEKU,colours,5,orange,English,1583.3,0.119329,1.003870,False


---
## 1. Subjects Whose All Responses Were in English

For each subject we check whether **every** word across all their domains was tagged `English`.

> **Finding:** No subject responded exclusively in English — every participant used at least one Hindi/Hinglish word. The code below still reports this and additionally identifies **English-dominant** subjects (< 30% Hindi words) as the closest group.

In [2]:
# ── 1. Fully-English subjects ────────────────────────────────────────────────
subj_lang = df.groupby("subject_id")["is_hindi"].any()
fully_english_subjects = subj_lang[~subj_lang].index.tolist()

print(f"Subjects with ALL responses in English: {len(fully_english_subjects)}")
if fully_english_subjects:
    print(fully_english_subjects)
else:
    print("None — every subject used at least one Hindi/Hinglish word.")

# Also compute per-subject Hindi %
subj_hindi_pct = (
    df.groupby("subject_id")["is_hindi"]
    .agg(total="count", hindi_words="sum")
    .assign(hindi_pct=lambda d: d["hindi_words"] / d["total"] * 100)
    .reset_index()
)

# English-dominant: < 30% Hindi (but not 0%)
english_dominant = subj_hindi_pct[
    (subj_hindi_pct["hindi_pct"] > 0) & (subj_hindi_pct["hindi_pct"] < 30)
].copy()

print(f"\nEnglish-dominant subjects (0% < Hindi < 30%) — {len(english_dominant)} subjects:")
display(english_dominant.sort_values("hindi_pct").round(1))

print(f"\nAll subjects ranked by Hindi %:")
display(subj_hindi_pct.sort_values("hindi_pct").round(1).to_string(index=False))

Subjects with ALL responses in English: 0
None — every subject used at least one Hindi/Hinglish word.

English-dominant subjects (0% < Hindi < 30%) — 8 subjects:


,subject_id,total,hindi_words,hindi_pct
29,83169,46,2,4.3
25,77358,32,2,6.2
14,49294,44,3,6.8
3,10255,50,6,12.0
10,34350,55,7,12.7
8,25803,42,6,14.3
24,76112,40,7,17.5
27,81788,44,10,22.7



All subjects ranked by Hindi %:


' subject_id  total  hindi_words  hindi_pct\n      83169     46            2        4.3\n      77358     32            2        6.2\n      49294     44            3        6.8\n      10255     50            6       12.0\n      34350     55            7       12.7\n      25803     42            6       14.3\n      76112     40            7       17.5\n      81788     44           10       22.7\n      53307     28           26       92.9\n      20970     31           29       93.5\n      92821     52           50       96.2\n       5157     28           27       96.4\n      23853     24           24      100.0\n      13549     26           26      100.0\n      10147     17           17      100.0\n      13498     20           20      100.0\n      53105     21           21      100.0\n      43909     29           29      100.0\n      42616     16           16      100.0\n      35389     32           32      100.0\n      31417     24           24      100.0\n      61476     31           31

In [3]:
# Visual: stacked bar — English vs Hindi word count per subject
lang_per_subj = (
    df.groupby(["subject_id", "language_type"])["word"]
    .count()
    .reset_index(name="count")
)

fig = px.bar(
    lang_per_subj,
    x="subject_id",
    y="count",
    color="language_type",
    color_discrete_map={"English": "#1f77b4", "Hindi/Hinglish": "#ff7f0e"},
    title="Word Count per Subject by Language",
    labels={"count": "Word Count", "subject_id": "Subject ID", "language_type": "Language"},
    barmode="stack",
    category_orders={"language_type": ["English", "Hindi/Hinglish"]},
)
fig.update_layout(xaxis_tickangle=45, height=450)
fig.show()

---
## 2. Domain-wise Average Position Where User Switches from Hindi/Hinglish to English

**Definition:** For each (subject, domain) pair, we find the **first English word** that appears **after** the user has already produced at least one Hindi/Hinglish word — i.e., the "Hindi → English" switch point.  
The *switch position* is the word-order position at which that first post-Hindi English word occurs.  
We then average these switch positions across subjects for each domain.

Subjects/domains where the user **never switches back to English** (stays in Hindi throughout) are excluded from the average but counted separately.

In [4]:
# ── 2. Hindi → English switch position ──────────────────────────────────────
def first_hindi_to_english_switch(group):
    """
    For a sorted (by position) sequence of words, return the position of
    the first English word that comes AFTER at least one Hindi/Hinglish word.
    Returns NaN if no such switch occurs.
    """
    group = group.sort_values("position")
    seen_hindi = False
    for _, row in group.iterrows():
        if row["is_hindi"]:
            seen_hindi = True
        elif seen_hindi:          # English word that follows a Hindi word
            return row["position"]
    return np.nan


switch_records = []
for (subj, dom), grp in df.groupby(["subject_id", "domain"]):
    sw = first_hindi_to_english_switch(grp)
    total_hindi = grp["is_hindi"].sum()
    has_hindi = total_hindi > 0
    switch_records.append({
        "subject_id": subj,
        "domain": dom,
        "switch_position": sw,
        "total_words": len(grp),
        "hindi_words": int(total_hindi),
        "has_hindi": has_hindi,
        "switched_to_english": not np.isnan(sw) if has_hindi else False,
    })

switch_df = pd.DataFrame(switch_records)

# Domain-wise averages (only for pairs where a switch actually happens)
domain_switch_avg = (
    switch_df[switch_df["switched_to_english"]]
    .groupby("domain")["switch_position"]
    .agg(["mean", "median", "count"])
    .round(2)
    .rename(columns={"mean": "avg_switch_position", "median": "median_switch_position", "count": "n_subjects"})
    .reset_index()
)

print("Domain-wise average position of first Hindi→English switch:\n")
display(domain_switch_avg)

# How many subjects-domain pairs had: only Hindi, only English, or mixed with switch
print("\nSwitching status breakdown per domain:")
status_table = (
    switch_df.assign(
        status=lambda d: np.where(
            ~d["has_hindi"], "English only",
            np.where(d["switched_to_english"], "Hindi→English switch", "Hindi only (no switch)")
        )
    )
    .groupby(["domain", "status"])["subject_id"]
    .count()
    .unstack(fill_value=0)
    .reset_index()
)
display(status_table)

Domain-wise average position of first Hindi→English switch:



,domain,avg_switch_position,median_switch_position,n_subjects
0,animals,4.67,4.0,3
1,body-parts,6.00,6.0,1
2,foods,7.00,5.0,10



Switching status breakdown per domain:


status,domain,English only,Hindi only (no switch),Hindi→English switch
0,animals,7,25,3
1,body-parts,1,22,1
2,colours,7,4,0
3,foods,0,25,10


In [5]:
# Visual: scatter strip + mean bar for switch positions per domain
switched_data = switch_df[switch_df["switched_to_english"]].copy()

# Strip plot via px.scatter with jitter
fig = px.scatter(
    switched_data,
    x="domain",
    y="switch_position",
    color="domain",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Hindi\u2192English Switch Position per Domain (word # at first switch)",
    labels={"switch_position": "Switch Position (word #)", "domain": "Domain"},
    hover_data=["subject_id"],
)

# Overlay mean as horizontal dash
for i, row in domain_switch_avg.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["domain"]],
        y=[row["avg_switch_position"]],
        mode="markers",
        marker=dict(symbol="line-ew", size=35, color="black", line=dict(width=3)),
        name="Mean" if i == 0 else None,
        showlegend=(i == 0),
    ))

fig.update_layout(height=450, showlegend=True)
fig.show()

# Bar chart of averages
fig2 = px.bar(
    domain_switch_avg,
    x="domain",
    y="avg_switch_position",
    text="avg_switch_position",
    color="domain",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Average Hindi\u2192English Switch Position by Domain",
    labels={"avg_switch_position": "Avg Switch Position (word #)", "domain": "Domain"},
    hover_data=["n_subjects", "median_switch_position"],
)
fig2.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig2.update_layout(showlegend=False, height=400)
fig2.show()

---
## 3. Overall Language Use Distribution

How much of the total vocabulary produced across subjects is in each language, broken down by domain?

In [6]:
# ── 3. Language distribution by domain ──────────────────────────────────────
domain_lang = (
    df.groupby(["domain", "language_type"])["word"]
    .count()
    .reset_index(name="count")
)
domain_lang["pct"] = domain_lang.groupby("domain")["count"].transform(lambda x: x / x.sum() * 100)

fig = px.bar(
    domain_lang,
    x="domain",
    y="pct",
    color="language_type",
    color_discrete_map={"English": "#1f77b4", "Hindi/Hinglish": "#ff7f0e"},
    barmode="stack",
    text=domain_lang["pct"].round(1).astype(str) + "%",
    title="Percentage of English vs Hindi/Hinglish Words by Domain",
    labels={"pct": "Percentage (%)", "domain": "Domain", "language_type": "Language"},
)
fig.update_traces(textposition="inside")
fig.update_layout(height=420, uniformtext_minsize=10)
fig.show()

print("\nRaw word counts per domain × language:")
display(domain_lang.pivot(index="domain", columns="language_type", values="count").fillna(0).astype(int))


Raw word counts per domain × language:


language_type,English,Hindi/Hinglish
domain,,
animals,132,233
body-parts,15,191
colours,106,41
foods,64,258


---
## 4. Subject-Level Language Profile Classification

Each subject is categorised into one of four profiles:
- **English-only** — all words in English
- **Hindi-dominant** — >70 % Hindi/Hinglish words
- **English-dominant** — >70 % English words (but not 100 %)
- **Balanced mixer** — between 30 % and 70 % Hindi/Hinglish

In [7]:
# ── 4. Subject language profile ──────────────────────────────────────────────
subj_profile = df.groupby("subject_id").agg(
    total=("word", "count"),
    hindi_count=("is_hindi", "sum"),
).reset_index()

subj_profile["hindi_pct"] = subj_profile["hindi_count"] / subj_profile["total"] * 100

def classify(pct):
    if pct == 0:
        return "English-only"
    elif pct >= 70:
        return "Hindi-dominant (≥70%)"
    elif pct > 30:
        return "Balanced mixer (30–70%)"
    else:
        return "English-dominant (<30% Hindi)"

subj_profile["profile"] = subj_profile["hindi_pct"].apply(classify)

profile_order = [
    "English-only",
    "English-dominant (<30% Hindi)",
    "Balanced mixer (30–70%)",
    "Hindi-dominant (≥70%)",
]

print("Subject language profiles:")
display(subj_profile[["subject_id", "total", "hindi_count", "hindi_pct", "profile"]].round(1))

fig = px.histogram(
    subj_profile,
    x="profile",
    category_orders={"profile": profile_order},
    color="profile",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Distribution of Subject Language Profiles",
    labels={"profile": "Language Profile", "count": "Number of Subjects"},
    text_auto=True,
)
fig.update_layout(showlegend=False, height=400, yaxis_title="Number of Subjects")
fig.show()

Subject language profiles:


,subject_id,total,hindi_count,hindi_pct,profile
0,3342,21,21,100.0,Hindi-dominant (≥70%)
1,5157,28,27,96.4,Hindi-dominant (≥70%)
2,10147,17,17,100.0,Hindi-dominant (≥70%)
3,10255,50,6,12.0,English-dominant (<30% Hindi)
4,13498,20,20,100.0,Hindi-dominant (≥70%)
5,13549,26,26,100.0,Hindi-dominant (≥70%)
6,20970,31,29,93.5,Hindi-dominant (≥70%)
7,23853,24,24,100.0,Hindi-dominant (≥70%)
8,25803,42,6,14.3,English-dominant (<30% Hindi)
9,31417,24,24,100.0,Hindi-dominant (≥70%)


---
## 5. Reaction Time: English vs Hindi/Hinglish Words

Do subjects respond faster when producing words in their dominant language?  
We compare mean `rt_ms` across language types overall and per domain.

In [8]:
# ── 5. Reaction time by language ─────────────────────────────────────────────
rt_stats = df.groupby("language_type")["rt_ms"].agg(["mean", "median", "std"]).round(1)
print("Overall RT stats by language:")
display(rt_stats)

# Box plot per domain × language
fig = px.box(
    df,
    x="domain",
    y="rt_ms",
    color="language_type",
    color_discrete_map={"English": "#1f77b4", "Hindi/Hinglish": "#ff7f0e"},
    title="Reaction Time Distribution by Domain and Language",
    labels={"rt_ms": "Reaction Time (ms)", "domain": "Domain", "language_type": "Language"},
    points="outliers",
)
fig.update_layout(height=450)
fig.show()

Overall RT stats by language:


,mean,median,std
language_type,,,
English,3771.8,2702.6,3323.0
Hindi/Hinglish,6373.0,5234.1,4924.7


---
## 6. Reaction Time at Language Switches

When a subject switches language (English → Hindi or Hindi → English), is there a spike in reaction time compared to same-language consecutive words?  
We compare the RT of "switch" words vs "non-switch" words.

In [9]:
# ── 6. RT at language switch points ─────────────────────────────────────────
switch_rt_records = []
for (subj, dom), grp in df.groupby(["subject_id", "domain"]):
    grp = grp.sort_values("position").reset_index(drop=True)
    for i, row in grp.iterrows():
        if i == 0:
            switch_rt_records.append({**row, "is_switch": False, "switch_dir": "none"})
        else:
            prev_lang = grp.loc[i - 1, "language_type"]
            curr_lang = row["language_type"]
            is_switch = prev_lang != curr_lang
            if is_switch:
                direction = "Eng→Hindi" if prev_lang == "English" else "Hindi→Eng"
            else:
                direction = "none"
            switch_rt_records.append({**row, "is_switch": is_switch, "switch_dir": direction})

switch_rt_df = pd.DataFrame(switch_rt_records)

# Compare RT: switch vs non-switch
rt_compare = switch_rt_df.groupby("is_switch")["rt_ms"].agg(["mean", "median", "count"]).round(1)
rt_compare.index = rt_compare.index.map({True: "Switch word", False: "Non-switch word"})
print("RT comparison — switch vs non-switch words:")
display(rt_compare)

# RT by switch direction
rt_by_dir = (
    switch_rt_df[switch_rt_df["is_switch"]]
    .groupby("switch_dir")["rt_ms"]
    .agg(["mean", "median", "count"])
    .round(1)
)
print("\nRT by switch direction:")
display(rt_by_dir)

# Visualise
fig = px.violin(
    switch_rt_df.assign(
        switch_label=switch_rt_df["is_switch"].map({True: "Switch word", False: "Non-switch word"})
    ),
    x="switch_label",
    y="rt_ms",
    color="switch_label",
    box=True,
    points="outliers",
    color_discrete_map={"Switch word": "#d62728", "Non-switch word": "#2ca02c"},
    title="Reaction Time: Switch Words vs Non-switch Words",
    labels={"rt_ms": "Reaction Time (ms)", "switch_label": ""},
)
fig.update_layout(showlegend=False, height=420)
fig.show()

RT comparison — switch vs non-switch words:


,mean,median,count
is_switch,,,
Non-switch word,5649.5,4520.6,997
Switch word,3973.4,3596.7,43



RT by switch direction:


,mean,median,count
switch_dir,,,
Eng→Hindi,4409.5,3567.7,22
Hindi→Eng,3516.5,3596.7,21


---
## 7. Word Output Volume: Total Words per Subject per Domain

Do bilingual/Hindi-dominant subjects produce fewer or more words?  
We plot total word count per (subject × domain), coloured by language profile.

In [10]:
# ── 7. Word output volume ────────────────────────────────────────────────────
vol_df = (
    df.groupby(["subject_id", "domain"])["word"]
    .count()
    .reset_index(name="word_count")
    .merge(subj_profile[["subject_id", "profile"]], on="subject_id")
)

# Box plot per domain, striped by profile
fig = px.strip(
    vol_df,
    x="domain",
    y="word_count",
    color="profile",
    color_discrete_sequence=px.colors.qualitative.Safe,
    stripmode="overlay",
    title="Total Words per Subject per Domain",
    labels={"word_count": "Total Words", "domain": "Domain", "profile": "Language Profile"},
    hover_data=["subject_id"],
)

# Overlay domain mean
dom_mean = vol_df.groupby("domain")["word_count"].mean().reset_index()
fig.add_trace(go.Scatter(
    x=dom_mean["domain"],
    y=dom_mean["word_count"],
    mode="markers",
    marker=dict(symbol="line-ew", size=35, color="black", line=dict(width=3)),
    name="Domain mean",
))
fig.update_layout(height=460)
fig.show()

# Stats table
print("\nAverage word count per domain × profile:")
display(
    vol_df.groupby(["domain", "profile"])["word_count"]
    .agg(["mean", "std", "count"])
    .round(2)
    .reset_index()
)


Average word count per domain × profile:


,domain,profile,mean,std,count
0,animals,English-dominant (<30% Hindi),16.50,4.54,8
1,animals,Hindi-dominant (≥70%),8.63,2.72,27
2,body-parts,English-dominant (<30% Hindi),14.00,NaN,1
3,body-parts,Hindi-dominant (≥70%),8.35,2.92,23
4,colours,English-dominant (<30% Hindi),15.14,1.86,7
5,colours,Hindi-dominant (≥70%),10.25,3.40,4
6,foods,English-dominant (<30% Hindi),12.62,3.34,8
7,foods,Hindi-dominant (≥70%),8.19,3.71,27


---
## 8. Language-Switch Heatmap Across Subjects and Domains

A heatmap of % Hindi/Hinglish words for every (subject × domain) combination gives a bird's-eye view of who switches where.

In [11]:
# ── 8. Hindi % heatmap (subject × domain) ────────────────────────────────────
heat_df = (
    df.groupby(["subject_id", "domain"])
    .agg(hindi_pct=("is_hindi", lambda x: x.mean() * 100))
    .reset_index()
    .pivot(index="subject_id", columns="domain", values="hindi_pct")
)

# Sort subjects by overall Hindi%
heat_df = heat_df.loc[subj_profile.set_index("subject_id")["hindi_pct"].sort_values(ascending=False).index]

fig = go.Figure(data=go.Heatmap(
    z=heat_df.values,
    x=heat_df.columns.tolist(),
    y=heat_df.index.tolist(),
    colorscale="RdYlGn_r",
    zmin=0, zmax=100,
    text=np.round(heat_df.values, 1),
    texttemplate="%{text}%",
    colorbar=dict(title="% Hindi/Hinglish"),
))
fig.update_layout(
    title="% Hindi/Hinglish Words per Subject × Domain (subjects sorted by overall Hindi%)",
    xaxis_title="Domain",
    yaxis_title="Subject ID",
    height=700,
    yaxis=dict(tickmode="linear"),
)
fig.show()

---
## 9. Positional Trend: Does Language Shift Later in the Sequence?

As subjects reach deeper into their response sequence (higher position), do they increasingly resort to Hindi/Hinglish (vocabulary exhaustion effect)?  
We bin positions into quintiles and look at average Hindi% per bin.

In [12]:
# ── 9. Positional language trend ─────────────────────────────────────────────
pos_trend = df.copy()
pos_trend["pos_bin"] = pd.cut(pos_trend["position"], bins=[0,3,6,9,12,100],
                               labels=["1–3","4–6","7–9","10–12","13+"])

trend = (
    pos_trend.groupby(["pos_bin", "language_type"])["word"]
    .count()
    .reset_index(name="count")
)

# Pivot to get Hindi%
trend_pct = (
    trend.pivot(index="pos_bin", columns="language_type", values="count")
    .fillna(0)
    .assign(hindi_pct=lambda d: d["Hindi/Hinglish"] / (d["Hindi/Hinglish"] + d["English"]) * 100)
    .reset_index()
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=trend_pct["pos_bin"].astype(str),
    y=trend_pct["hindi_pct"].round(1),
    marker_color="#ff7f0e",
    text=trend_pct["hindi_pct"].round(1).astype(str) + "%",
    textposition="outside",
    name="Hindi/Hinglish %",
))
fig.update_layout(
    title="% Hindi/Hinglish Words by Ordinal Position Bin",
    xaxis_title="Word Position Bin",
    yaxis_title="% Hindi/Hinglish",
    yaxis_range=[0, 100],
    height=400,
)
fig.show()

# Per domain positional trend
trend_dom = (
    pos_trend.groupby(["pos_bin", "domain", "language_type"])["word"]
    .count()
    .reset_index(name="count")
)
trend_dom_pct = (
    trend_dom.pivot_table(index=["pos_bin", "domain"], columns="language_type", values="count", fill_value=0)
    .reset_index()
    .assign(hindi_pct=lambda d: d["Hindi/Hinglish"] / (d["Hindi/Hinglish"] + d.get("English", 0)) * 100)
)

fig2 = px.line(
    trend_dom_pct,
    x="pos_bin",
    y="hindi_pct",
    color="domain",
    markers=True,
    title="% Hindi/Hinglish by Position Bin — per Domain",
    labels={"hindi_pct": "% Hindi/Hinglish", "pos_bin": "Position Bin", "domain": "Domain"},
)
fig2.update_layout(height=420)
fig2.show()

---
## 10. Top Words per Domain × Language

What are the most frequently produced words in each domain, split by language?

In [13]:
# ── 10. Top words per domain × language ──────────────────────────────────────
domains = df["domain"].unique()
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[f"{dom} – English" for dom in domains] + [f"{dom} – Hindi/Hinglish" for dom in domains],
)

for col_i, dom in enumerate(domains, start=1):
    for row_i, lang in enumerate(["English", "Hindi/Hinglish"], start=1):
        sub = (
            df[(df["domain"] == dom) & (df["language_type"] == lang)]["word"]
            .str.lower()
            .value_counts()
            .head(8)
            .reset_index()
        )
        sub.columns = ["word", "count"]
        color = "#1f77b4" if lang == "English" else "#ff7f0e"
        fig.add_trace(
            go.Bar(x=sub["word"], y=sub["count"], marker_color=color, showlegend=False),
            row=row_i, col=col_i,
        )

fig.update_layout(
    title_text="Top 8 Words per Domain × Language",
    height=600,
    showlegend=False,
)
fig.show()

---
---
# Part 2 — Enriched EDA with Demographic Data

`vft_responses_enriched.csv` joins VFT word-level data with participant demographics extracted from `responses.json`.

**New fields available per subject:**

| Field | Description |
|---|---|
| `age` | Participant age (years) |
| `gender` | Gender |
| `education` | Years of formal education |
| `state_ut` | State / UT of origin |
| `first_language` | Mother tongue |
| `language_count` | Number of languages known |
| `Hi_Read` / `Hi_Write` | Hindi reading / writing proficiency (1–5) |
| `En_Read` / `En_Write` | English reading / writing proficiency (1–5) |
| `hi_confidence` | Self-rated Hindi confidence (1–5) |
| `en_confidence` | Self-rated English confidence (1–5) |
| `hi_acquisition_age` | Age at which Hindi was acquired |
| `en_acquisition_age` | Age at which English was acquired |
| `dominant_hand` | Handedness |
| `alert_time` | Peak alertness time of day |

In [14]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Load enriched dataset
edf = pd.read_csv("vft_responses_enriched.csv")
edf["is_hindi"] = edf["language_type"] == "Hindi/Hinglish"

# Some demographic columns may be absent depending on how the file was generated.
# Add missing columns as NaN so downstream analysis still runs safely.
expected_demo_cols = [
    "hi_confidence", "en_confidence",
    "hi_acquisition_age", "en_acquisition_age",
    "dominant_hand", "alert_time",
]
missing_demo_cols = [c for c in expected_demo_cols if c not in edf.columns]
for c in missing_demo_cols:
    edf[c] = np.nan
if missing_demo_cols:
    print("Missing columns in CSV (filled with NaN):", missing_demo_cols)

# Subject-level summary (one row per subject)
subj = (
    edf.groupby("subject_id")
    .agg(
        total_words=("word", "count"),
        hindi_words=("is_hindi", "sum"),
        avg_rt_ms=("rt_ms", "mean"),
        age=("age", "first"),
        gender=("gender", "first"),
        education=("education", "first"),
        state_ut=("state_ut", "first"),
        first_language=("first_language", "first"),
        language_count=("language_count", "first"),
        Hi_Read=("Hi_Read", "first"),
        Hi_Write=("Hi_Write", "first"),
        En_Read=("En_Read", "first"),
        En_Write=("En_Write", "first"),
        hi_confidence=("hi_confidence", "first"),
        en_confidence=("en_confidence", "first"),
        hi_acquisition_age=("hi_acquisition_age", "first"),
        en_acquisition_age=("en_acquisition_age", "first"),
        dominant_hand=("dominant_hand", "first"),
        alert_time=("alert_time", "first"),
    )
    .reset_index()
)
subj["hindi_pct"] = subj["hindi_words"] / subj["total_words"] * 100
subj["hi_proficiency"] = (subj["Hi_Read"] + subj["Hi_Write"]) / 2
subj["en_proficiency"] = (subj["En_Read"] + subj["En_Write"]) / 2

# Normalise free-text fields (capitalise, fix obvious typos)
subj["first_language"] = subj["first_language"].str.strip().str.title()
subj["first_language"] = subj["first_language"].replace({"Gujarti": "Gujarati", "Gujaati": "Gujarati", "Gujarati": "Gujarati"})
edf["first_language"] = edf["first_language"].str.strip().str.title()
edf["first_language"] = edf["first_language"].replace({"Gujarti": "Gujarati", "Gujaati": "Gujarati"})

print("Enriched subject table shape:", subj.shape)
print("\nDemographic summary:")
display(subj[["subject_id","age","gender","education","state_ut","first_language",
              "language_count","hindi_pct"]].head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'vft_responses_enriched.csv'

---
## 11. Demographic Overview of Participants

Basic distributions: age, gender, education, handedness, and alert time.

In [ ]:
# ── 11. Demographic overview ─────────────────────────────────────────────────
print("Age stats:", subj["age"].describe().round(1).to_dict())
print("Gender counts:\n", subj["gender"].value_counts().to_string())
print("\nEducation stats:", subj["education"].describe().round(1).to_dict())
print("\nDominant hand:\n", subj["dominant_hand"].value_counts().to_string())
print("\nAlert time:\n", subj["alert_time"].value_counts().to_string())

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Age Distribution", "Education (years)", "Gender"],
    specs=[[{"type": "histogram"}, {"type": "histogram"}, {"type": "pie"}]],
)

fig.add_trace(go.Histogram(x=subj["age"], nbinsx=10, marker_color="#1f77b4", name="Age"), row=1, col=1)
fig.add_trace(go.Histogram(x=subj["education"], nbinsx=10, marker_color="#2ca02c", name="Education"), row=1, col=2)
fig.add_trace(
    go.Pie(labels=subj["gender"].value_counts().index, values=subj["gender"].value_counts().values,
           name="Gender", hole=0.4),
    row=1, col=3,
)

fig.update_layout(title="Participant Demographics", height=380, showlegend=False)
fig.show()

---
## 12. Hindi Usage by State of Origin

Does where a participant is from predict how much Hindi/Hinglish they use in VFT?  
States are sorted by median Hindi%.

In [ ]:
# ── 12. Hindi% by state ──────────────────────────────────────────────────────
state_order = (
    subj.groupby("state_ut")["hindi_pct"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

fig = px.strip(
    subj,
    x="state_ut",
    y="hindi_pct",
    color="first_language",
    hover_data=["subject_id", "age", "education"],
    title="% Hindi/Hinglish by State of Origin (coloured by Mother Tongue)",
    labels={"hindi_pct": "% Hindi/Hinglish", "state_ut": "State / UT", "first_language": "Mother Tongue"},
    category_orders={"state_ut": state_order},
    stripmode="overlay",
)
fig.update_layout(height=500, xaxis_tickangle=40)
fig.show()

print("\nSubjects per state:")
display(
    subj.groupby("state_ut")
    .agg(n_subjects=("subject_id","count"), median_hindi_pct=("hindi_pct","median"), mean_hindi_pct=("hindi_pct","mean"))
    .round(1)
    .sort_values("median_hindi_pct", ascending=False)
    .reset_index()
)

---
## 13. Mother Tongue vs Hindi Usage

Does having Hindi as a first language predict higher Hindi% in VFT responses?

In [ ]:
# ── 13. Mother tongue vs Hindi% ──────────────────────────────────────────────
lang_order = (
    subj.groupby("first_language")["hindi_pct"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig = px.box(
    subj,
    x="first_language",
    y="hindi_pct",
    color="first_language",
    points="all",
    hover_data=["subject_id", "state_ut"],
    title="Hindi% by Mother Tongue",
    labels={"hindi_pct": "% Hindi/Hinglish", "first_language": "Mother Tongue"},
    category_orders={"first_language": lang_order},
)
fig.update_layout(showlegend=False, height=440, xaxis_tickangle=30)
fig.show()

print("\nStats per mother tongue:")
display(
    subj.groupby("first_language")
    .agg(n=("subject_id","count"), median_hindi=("hindi_pct","median"), mean_hindi=("hindi_pct","mean"))
    .round(1)
    .sort_values("median_hindi", ascending=False)
)

---
## 14. Hindi Proficiency vs Actual Hindi Usage

Self-rated Hindi proficiency (`Hi_Read` + `Hi_Write` average) vs the actual % of Hindi/Hinglish words produced.  
We also check whether self-rated English proficiency is inversely correlated with Hindi usage.

In [ ]:
# ── 14. Proficiency vs Hindi usage ───────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Hindi Proficiency vs Hindi%",
        "English Proficiency vs Hindi%",
    ],
)

# Hindi proficiency
fig.add_trace(
    go.Scatter(
        x=subj["hi_proficiency"], y=subj["hindi_pct"],
        mode="markers+text",
        text=subj["subject_id"].astype(str),
        textposition="top center",
        marker=dict(color=subj["hindi_pct"], colorscale="RdYlGn_r", size=10, showscale=False),
        name="Hindi prof.",
    ),
    row=1, col=1,
)

# English proficiency
fig.add_trace(
    go.Scatter(
        x=subj["en_proficiency"], y=subj["hindi_pct"],
        mode="markers+text",
        text=subj["subject_id"].astype(str),
        textposition="top center",
        marker=dict(color=subj["hindi_pct"], colorscale="RdYlGn_r", size=10, showscale=False),
        name="English prof.",
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text="Hindi Proficiency (avg Read+Write, 1–5)", row=1, col=1)
fig.update_xaxes(title_text="English Proficiency (avg Read+Write, 1–5)", row=1, col=2)
fig.update_yaxes(title_text="% Hindi/Hinglish", row=1, col=1)
fig.update_layout(height=430, showlegend=False,
                  title="Self-rated Language Proficiency vs Actual Hindi Usage")
fig.show()

# Correlation table
corr_cols = ["hi_proficiency", "en_proficiency", "hi_confidence", "en_confidence",
             "hi_acquisition_age", "en_acquisition_age", "age", "education", "language_count"]
print("\nPearson correlations with hindi_pct:")
corr = subj[corr_cols + ["hindi_pct"]].corr()["hindi_pct"].drop("hindi_pct").sort_values()
display(corr.round(3).to_frame("r"))

---
## 15. Education & Age vs Hindi Usage

Do older or less-educated participants rely more on Hindi/Hinglish?

In [ ]:
# ── 15. Education & Age vs Hindi% ────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2, subplot_titles=["Age vs Hindi%", "Education (years) vs Hindi%"])

for col_i, (xcol, xlabel) in enumerate([("age", "Age (years)"), ("education", "Education (years)")], 1):
    fig.add_trace(
        go.Scatter(
            x=subj[xcol], y=subj["hindi_pct"],
            mode="markers",
            marker=dict(
                size=10,
                color=subj["hindi_pct"],
                colorscale="RdYlGn_r",
                showscale=(col_i == 2),
                colorbar=dict(title="Hindi%", x=1.02),
            ),
            text=subj["first_language"],
            hovertemplate=f"{xlabel}: %{{x}}<br>Hindi%: %{{y:.1f}}<br>L1: %{{text}}<extra></extra>",
            name=xlabel,
            showlegend=False,
        ),
        row=1, col=col_i,
    )
    fig.update_xaxes(title_text=xlabel, row=1, col=col_i)
    fig.update_yaxes(title_text="% Hindi/Hinglish", row=1, col=col_i)

fig.update_layout(title="Age & Education vs Hindi Usage", height=430)
fig.show()

---
## 16. Hindi Acquisition Age vs Usage

Does acquiring Hindi earlier (lower `hi_acquisition_age`) predict higher Hindi% in VFT?  
Age of acquisition is coded as: 1 = birth / early childhood, 2 = school, 3 = later.

In [ ]:
# ── 16. Hindi acquisition age vs Hindi% ──────────────────────────────────────
acq_label = {1.0: "Early (birth)", 2.0: "School age", 3.0: "Late"}
subj["hi_acq_label"] = subj["hi_acquisition_age"].map(acq_label).fillna("Unknown")
acq_order = ["Early (birth)", "School age", "Late"]

fig = px.box(
    subj,
    x="hi_acq_label",
    y="hindi_pct",
    color="hi_acq_label",
    points="all",
    hover_data=["subject_id", "first_language", "state_ut"],
    title="Hindi Acquisition Age vs Hindi% in VFT",
    labels={"hindi_pct": "% Hindi/Hinglish", "hi_acq_label": "Hindi Acquisition Stage"},
    category_orders={"hi_acq_label": acq_order},
)
fig.update_layout(showlegend=False, height=420)
fig.show()

print("\nMean Hindi% by acquisition stage:")
display(
    subj.groupby("hi_acq_label")["hindi_pct"]
    .agg(["mean", "median", "count"])
    .round(1)
    .reindex([l for l in acq_order if l in subj["hi_acq_label"].unique()])
)

---
## 17. Number of Languages Known vs Hindi Usage & Word Output

Multilingual participants might code-switch more freely. Does knowing more languages predict more Hindi use or more words overall?

In [ ]:
# ── 17. Language count vs Hindi% and word output ─────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["# Languages Known vs Hindi%", "# Languages Known vs Total Words"],
)

for col_i, (ycol, ylabel) in enumerate([("hindi_pct", "% Hindi/Hinglish"), ("total_words", "Total Words")], 1):
    fig.add_trace(
        go.Box(
            x=subj["language_count"].astype(int),
            y=subj[ycol],
            boxpoints="all",
            jitter=0.3,
            pointpos=-1.6,
            marker_size=7,
            name=ylabel,
            showlegend=False,
        ),
        row=1, col=col_i,
    )
    fig.update_xaxes(title_text="Number of Languages Known", row=1, col=col_i)
    fig.update_yaxes(title_text=ylabel, row=1, col=col_i)

fig.update_layout(title="Multilingualism vs Language Use & Output", height=430)
fig.show()

print("\nMean Hindi% and total words by language count:")
display(
    subj.groupby("language_count")[["hindi_pct", "total_words"]]
    .agg(["mean", "count"])
    .round(1)
)

---
## 18. Reaction Time vs Education & Hindi Proficiency

Do participants with higher education or stronger Hindi proficiency respond faster in their chosen language?

In [ ]:
# ── 18. RT vs education & proficiency ────────────────────────────────────────
# Merge word-level RT back with subject demographics
rt_demo = edf.merge(
    subj[["subject_id","education","hi_proficiency","en_proficiency","gender","age","first_language"]],
    on="subject_id",
    how="left",
)

# Mean RT per subject per language
rt_subj = (
    rt_demo.groupby(["subject_id", "language_type"])["rt_ms"]
    .mean()
    .reset_index()
    .merge(subj[["subject_id","education","hi_proficiency","en_proficiency","hindi_pct","age"]], on="subject_id")
)

fig = px.scatter(
    rt_subj,
    x="education",
    y="rt_ms",
    color="language_type",
    size="hindi_pct",
    size_max=18,
    hover_data=["subject_id","age","hi_proficiency"],
    color_discrete_map={"English": "#1f77b4", "Hindi/Hinglish": "#ff7f0e"},
    facet_col="language_type",
    trendline="ols",
    title="Mean RT per Subject vs Education (bubble size = Hindi%)",
    labels={"rt_ms": "Mean RT (ms)", "education": "Education (years)", "language_type": "Language"},
)
fig.update_layout(height=430, showlegend=False)
fig.show()

---
## 19. Domain-wise Language Use by Demographics

Does the choice of language in a specific domain (e.g., `foods`) vary by mother tongue or state?  
Heatmap: median Hindi% per (first_language × domain).

In [ ]:
# ── 19. Domain × first_language Hindi% heatmap ───────────────────────────────
dom_lang_heat = (
    edf.groupby(["first_language", "domain"])["is_hindi"]
    .mean()
    .mul(100)
    .reset_index(name="hindi_pct")
    .pivot(index="first_language", columns="domain", values="hindi_pct")
)

fig = go.Figure(data=go.Heatmap(
    z=dom_lang_heat.values,
    x=dom_lang_heat.columns.tolist(),
    y=dom_lang_heat.index.tolist(),
    colorscale="RdYlGn_r",
    zmin=0, zmax=100,
    text=np.round(dom_lang_heat.values, 1),
    texttemplate="%{text}%",
    colorbar=dict(title="% Hindi"),
))
fig.update_layout(
    title="% Hindi/Hinglish per Mother Tongue × Domain",
    xaxis_title="Domain",
    yaxis_title="Mother Tongue",
    height=480,
)
fig.show()

# Also: bar chart of avg words per domain split by whether Hindi is mother tongue
edf["is_hindi_L1"] = edf["first_language"] == "Hindi"
dom_fluency = (
    edf.groupby(["subject_id", "domain", "is_hindi_L1"])["word"]
    .count()
    .reset_index(name="word_count")
)

fig2 = px.box(
    dom_fluency,
    x="domain",
    y="word_count",
    color="is_hindi_L1",
    points="all",
    color_discrete_map={True: "#d62728", False: "#1f77b4"},
    title="Word Output Volume per Domain: Hindi L1 vs Other L1",
    labels={"word_count": "Words Produced", "domain": "Domain", "is_hindi_L1": "Hindi is Mother Tongue"},
)
fig2.update_layout(height=420)
fig2.show()